# 💬 Meta AgentCore Chat — Test Notebook

Test the deployed chat agent from Python. Run `cdk deploy` first.

In [ ]:
import requests
import json
import boto3

# ✏️ Set these from CDK outputs (run `python update_ios_config.py --skip-deploy` to get them)
API_URL = "https://4b6v70k6e0.execute-api.us-east-1.amazonaws.com/prod"
APP_CLIENT_ID = "3tjqlt2a41vp3i8pnadknuh45s"
REGION = "us-east-1"

# ✏️ Your Cognito credentials (same as iOS app login)
USERNAME = "your@email.com"
PASSWORD = "yourPassword123"

DEVICE_ID = "test-notebook-001"
MEMORY_DEVICE_ID = "test-memory-001"  # Dedicated device_id for memory tests


def get_cognito_token() -> str:
    """Get a Cognito IdToken using USER_PASSWORD_AUTH."""
    client = boto3.client("cognito-idp", region_name=REGION)
    response = client.initiate_auth(
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters={"USERNAME": USERNAME, "PASSWORD": PASSWORD},
        ClientId=APP_CLIENT_ID,
    )
    token = response["AuthenticationResult"]["IdToken"]
    print(f"✅ Cognito token obtained (expires in {response['AuthenticationResult']['ExpiresIn']}s)")
    return token

TOKEN = get_cognito_token()

In [ ]:
def chat(prompt: str, device_id: str = DEVICE_ID) -> str:
    """Send a message to the chat agent using Cognito auth."""
    r = requests.post(
        f"{API_URL}/chat",
        headers={
            "Content-Type": "application/json",
            "Authorization": TOKEN,
        },
        json={"prompt": prompt, "device_id": device_id},
        timeout=120,
    )
    r.raise_for_status()
    data = r.json()
    print(f"📨 {prompt}")
    print(f"🤖 {data.get('response', data)}")
    print(f"   message_id: {data.get('message_id', 'N/A')}")
    print()
    return data.get("response", "")

## 🧪 Test 1: Basic conversation

In [ ]:
chat("Hello! What can you help me with?")

## 🧮 Test 2: Calculator tool

In [ ]:
chat("What is 15% tip on a $127.50 dinner bill? And what's the total?")

## 🕐 Test 3: Current time tool

In [ ]:
chat("What day and time is it right now?")

## 🔍 Test 4: Tavily web search

In [ ]:
chat("What are the latest features announced for Amazon Bedrock AgentCore?")

## 🧠 Test 5: Think tool (complex reasoning)

In [ ]:
chat("I have a meeting in Tokyo at 3pm JST and another in New York at 10am EST. Can I make both if I fly from Tokyo to New York? How long is the flight?")

## 🌐 Test 6: HTTP request tool

In [ ]:
chat("Make a GET request to https://api.github.com and tell me the current GitHub API version.")

## 💬 Test 7: Multi-turn conversation (context retention)

In [ ]:
chat("My name is Alex and I'm planning a trip to Japan.")

In [ ]:
chat("What's the best time of year to visit? And remind me, what's my name?")

In [ ]:
import time
time.sleep(5)  # Give memory time to consolidate

# Refresh token to simulate a new session
TOKEN = get_cognito_token()

# Same MEMORY_DEVICE_ID — memory is keyed by device_id
chat("What do you know about me?", device_id=MEMORY_DEVICE_ID)
chat("Based on my preferences, how would you recommend I learn about AWS services?", device_id=MEMORY_DEVICE_ID)

## 🧠 Test 9: Long-term memory — retrieve across sessions

Use a **new token** (simulate a new session) and ask what the agent remembers.
The agent should recall the facts from Test 8 without being told again.

> Note: AgentCore Memory consolidates asynchronously — wait a few seconds after Test 8 before running this.

In [ ]:
chat("My name is Alex. I live in Miami and I work as a software engineer.", device_id=MEMORY_DEVICE_ID)
chat("I prefer short and direct answers, and I like to know the source of information.", device_id=MEMORY_DEVICE_ID)

## 🧠 Test 8: Long-term memory — store facts

Share personal info. The agent should store it in AgentCore Memory.

In [ ]:
# ✏️ Set from CDK outputs: ObsidianApiUrl and ObsidianApiKeyId
OBSIDIAN_API_URL = "<ObsidianApiUrl>"
OBSIDIAN_API_KEY_ID = "<ObsidianApiKeyId>"

# Fetch the actual key value
api_gw = boto3.client("apigateway", region_name=REGION)
OBSIDIAN_API_KEY = api_gw.get_api_key(apiKey=OBSIDIAN_API_KEY_ID, includeValue=True)["value"]
print(f"✅ Obsidian API key retrieved")

# Call the Obsidian Gateway API
r = requests.post(
    f"{OBSIDIAN_API_URL}ideas",
    headers={"Content-Type": "application/json", "x-api-key": OBSIDIAN_API_KEY},
    json={"prompt": "I want to build a CLI tool that converts voice notes into structured meeting summaries"},
    timeout=120,
)
r.raise_for_status()
data = r.json()
print(f"🤖 {data.get('result', data)}")

## 📓 Test 11: Obsidian Gateway API (external callers)

This path goes through the full Obsidian Gateway Runtime — two LLM calls.
Use for external agents, scripts, or automation (not the iOS app).

In [ ]:
# Verify the note was saved in S3
import boto3
OBSIDIAN_BUCKET = "obsidian-vault-elicosas"  # ✏️ change if different

s3 = boto3.client("s3", region_name=REGION)
response = s3.list_objects_v2(Bucket=OBSIDIAN_BUCKET, Prefix="Ideas/")
ideas = response.get("Contents", [])
print(f"📚 {len(ideas)} ideas saved in S3:")
for obj in sorted(ideas, key=lambda x: x["LastModified"], reverse=True)[:5]:
    print(f"  - {obj['Key']} ({obj['LastModified'].strftime('%Y-%m-%d %H:%M')})")

In [ ]:
# Test: agent saves idea directly to S3 (single LLM call)
chat("Save this idea: build a tool that listens to my meetings and automatically creates action items in Obsidian")

## 📓 Test 10: Obsidian — save idea via voice (direct S3 write)

The agent structures the idea itself (title, summary, problem, solution, next steps)
and writes directly to S3. Single LLM call — optimized for iOS voice UX.